# SciPy trajectory optimization

**Learning goals:** transcribe a bounded double-integrator problem, solve it with SLSQP, and inspect feasibility separately from objective value.

**Predict first:** which acceleration bounds should become active when the goal is to reach the target with little control effort over a short horizon?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

intervals, dt, target = 30, 0.1, 1.0

def unpack(z):
    position = z[: intervals + 1]
    velocity = z[intervals + 1 : 2 * (intervals + 1)]
    acceleration = z[2 * (intervals + 1) :]
    return position, velocity, acceleration

def objective(z):
    _, _, acceleration = unpack(z)
    return dt * np.sum(acceleration**2)

def defects(z):
    position, velocity, acceleration = unpack(z)
    dynamics = np.r_[
        position[1:] - position[:-1] - dt * velocity[:-1] - 0.5 * dt**2 * acceleration,
        velocity[1:] - velocity[:-1] - dt * acceleration,
    ]
    boundary = [position[0], velocity[0], position[-1] - target, velocity[-1]]
    return np.r_[dynamics, boundary]

size = 2 * (intervals + 1) + intervals
guess = np.zeros(size)
guess[: intervals + 1] = np.linspace(0, target, intervals + 1)
bounds = [(None, None)] * (2 * (intervals + 1)) + [(-1.0, 1.0)] * intervals
result = minimize(objective, guess, method="SLSQP", bounds=bounds,
                  constraints={"type": "eq", "fun": defects},
                  options={"ftol": 1e-10, "maxiter": 1000})
position, velocity, acceleration = unpack(result.x)
print(result.message)
print("Maximum defect:", np.max(np.abs(defects(result.x))))

In [ ]:
time = np.arange(intervals + 1) * dt
fig, axes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)
axes[0].plot(time, position)
axes[0].set_ylabel("position")
axes[1].plot(time, velocity)
axes[1].set_ylabel("velocity")
axes[2].step(time[:-1], acceleration, where="post")
axes[2].axhline(1, color="black", linestyle=":")
axes[2].axhline(-1, color="black", linestyle=":")
axes[2].set(xlabel="time", ylabel="acceleration")
fig.tight_layout()
plt.show()

**Experiment:** reduce the horizon or tighten the acceleration bounds. Predict feasibility before solving, then use the maximum defect—not only `result.success`—to evaluate the answer.

In [ ]:
assert result.success
assert np.max(np.abs(defects(result.x))) < 1e-6
assert np.max(np.abs(acceleration)) <= 1.0 + 1e-9
assert abs(position[-1] - target) < 1e-8 and abs(velocity[-1]) < 1e-8
print("Checks passed.")